# BODAQS Macro Analysis Dashboard

This notebook drives the extracted backend (`bodaqs_analysis`) with minimal editing.

Workflow:
1. Load CSV
2. Normalize (zero + scale)
3. Estimate velocity/acceleration
4. Load event schema + detect events
5. Inspect metrics / quick plots


In [1]:
import pandas as pd
import numpy as np
import logging
from pathlib import Path

from bodaqs_analysis import run_macro
from bodaqs_analysis import normalize_and_scale
from bodaqs_analysis import estimate_va
from bodaqs_analysis import load_event_schema
from bodaqs_analysis import detect_events_from_schema
from bodaqs_analysis import extract_metrics_df

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)


In [2]:
# ---- Inputs ----
CSV_PATH = r"C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-04*.CSV"   # change to your log
SCHEMA_PATH = r"event schema\event_schema.yaml"

# Columns and full-ranges (mm) for normalization (edit as needed)
NORMALIZE_RANGES = {
    "front_shock [mm]": 170.0,
    "rear_shock [mm]": 55.0,
}

SAMPLE_RATE_HZ = 200   # if known; used for VA estimation when time col is absent


In [3]:
# ---- Batch orchestration ----

pattern = Path(CSV_PATH)
CSV_FILES = sorted(pattern.parent.glob(pattern.name))

if not CSV_FILES:
    raise FileNotFoundError(f"No CSV files matched: {CSV_PATH}")

logger.info(f"Found {len(CSV_FILES)} files:")
for p in CSV_FILES:
    logger.info("  %s", p.name)

events_all = []
metrics_all = []
results_all = []  # optional if you want to inspect per-file outputs

for p in CSV_FILES:
    logger.info(f"Processing {p.name} ...")
    results = run_macro(
        str(p),
        SCHEMA_PATH,
        normalize_ranges=NORMALIZE_RANGES,
        sample_rate_hz=SAMPLE_RATE_HZ,
    )
    results_all.append((p, results))  # optional

    # pull from the correct keys
    ev = results["events"]
    mt = results["metrics"]

    # only append if non-empty (optional, but handy)
    if isinstance(ev, pd.DataFrame) and not ev.empty:
        events_all.append(ev)
    if isinstance(mt, pd.DataFrame) and not mt.empty:
        metrics_all.append(mt)

events_df  = pd.concat(events_all, ignore_index=True) if events_all else pd.DataFrame()
metrics_df = pd.concat(metrics_all, ignore_index=True) if metrics_all else pd.DataFrame()

logging.debug("events_df: %s", events_df.shape)
logging.debug("metrics_df: %s", metrics_df.shape)






INFO:__main__:Found 2 files:
INFO:__main__:  2026-01-04_06-37-32.CSV
INFO:__main__:  2026-01-04_07-08-37.CSV
INFO:__main__:Processing 2026-01-04_06-37-32.CSV ...
INFO:bodaqs_analysis.pipeline:Session load complete: C:\Users\Ben Connor\OneDrive\Documents\logs\2026-01-04_06-37-32.CSV
INFO:bodaqs_analysis.pipeline:Session pre-process complete
INFO:bodaqs_analysis.pipeline:Schema load complete


meta keys: ['channel_info', 'channels', 'device', 'notes', 'sample_rate_by_channel_hz', 'sample_rate_hz', 'session_id', 'signals', 'streams']
meta.session_id: 2026-01-04_06-37-32
df has rear_shock [mm]?: False
rear-ish: ['rear_shock_dom_suspension [mm]', 'rear_shock_raw_dom_suspension [counts]', 'rear_shock_dom_suspension [mm]_op_zeroed', 'rear_shock_norm [1]', 'rear_shock_vel [mm/s]', 'rear_shock_acc [mm/s^2]']


KeyError: "Event 'deep_compression': trigger.signal 'disp' maps to column 'rear_shock [mm]', which is not in event_analysis_df."

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output


def metric_histogram_widget_v5(
    events_df: pd.DataFrame,
    metrics_df: pd.DataFrame,
    *,
    event_type_col: str = "event_name",
    signal_col: str = "signal_col",
    default_bins: int = 10,
    max_bins: int = 200,
):
    logging.debug("events_df shape: %s", events_df.shape)
    logging.debug("metrics_df shape: %s", metrics_df.shape)

    for col in ("session_id", "event_id", event_type_col, signal_col):
        if col not in events_df.columns:
            raise ValueError(f"events_df missing required column: {col}")
    for col in ("session_id", "event_id"):
        if col not in metrics_df.columns:
            raise ValueError(f"metrics_df missing required column: {col}")

    metric_cols = [c for c in metrics_df.columns if isinstance(c, str) and c.startswith("m_")]
    if not metric_cols:
        raise ValueError("No metric columns found in metrics_df (expected 'm_*')")

    viz_df = events_df[["session_id", "event_id", event_type_col, signal_col]].merge(
        metrics_df[["session_id", "event_id"] + metric_cols],
        on=["session_id", "event_id"],
        how="inner",
        validate="one_to_one",
    )
    print("joined viz_df shape:", viz_df.shape)

    sessions = sorted([x for x in viz_df["session_id"].dropna().unique()])
    event_types = sorted([x for x in viz_df[event_type_col].dropna().unique()])
    signals = sorted([x for x in viz_df[signal_col].dropna().unique()])
    metrics = sorted(metric_cols)

    if not sessions:
        raise ValueError("No non-null session_id values after join")
    if not event_types:
        raise ValueError(f"No non-null values found in {event_type_col} after join")
    if not signals:
        raise ValueError(f"No non-null values found in {signal_col} after join")

    # --- widgets ---
    w_sess_mode = W.RadioButtons(
        options=[("Aggregate sessions", False), ("Compare sessions", True)],
        value=False,
        description="Sessions:",
    )
    w_sessions = W.SelectMultiple(
        options=sessions,
        value=tuple(sessions[:1]),
        description="Pick:",
        rows=min(8, max(3, len(sessions))),
    )

    w_sig_mode = W.RadioButtons(
        options=[("Aggregate signals", False), ("Compare signals", True)],
        value=False,
        description="Signals:",
    )
    w_signals = W.SelectMultiple(
        options=signals,
        value=tuple(signals[:1]),
        description="Pick:",
        rows=min(8, max(3, len(signals))),
    )

    w_event = W.Dropdown(options=event_types, value=event_types[0], description="Event:")
    w_metric = W.Dropdown(options=metrics, value=metrics[0], description="Metric:")
    w_bins = W.BoundedIntText(value=int(default_bins), min=1, max=int(max_bins), step=1, description="Bins:")
    w_cdf = W.Checkbox(value=False, description="CDF (cumulative)")
    w_norm = W.Checkbox(value=True, description="Normalize (proportion)")

    out = W.Output()

    def _series_stats(name: str, vals: np.ndarray) -> str:
        if len(vals) == 0:
            return f"- {name}: count=0"
        vmin = float(np.min(vals))
        vmax = float(np.max(vals))
        mean = float(np.mean(vals))
        med = float(np.median(vals))
        return f"- {name}: count={len(vals)}  min={vmin:.6g}  max={vmax:.6g}  mean={mean:.6g}  median={med:.6g}"

    def _plot_series(ax, vals: np.ndarray, bins: int, *, label: str | None):
        if len(vals) == 0:
            return
        if w_cdf.value:
            x = np.sort(vals)
            y = np.arange(1, len(x) + 1, dtype=float)
            if w_norm.value:
                y = y / float(len(x))
            ax.step(x, y, where="post", label=label)
        else:
            weights = None
            if w_norm.value:
                weights = np.ones_like(vals, dtype=float) / float(len(vals))
            ax.hist(vals, bins=int(bins), weights=weights, histtype="step", label=label)

    def _render(*_):
        with out:
            clear_output(wait=True)

            sel_sessions = list(w_sessions.value)
            sel_signals = list(w_signals.value)

            if not sel_sessions:
                print("Select at least one session.")
                return
            if not sel_signals:
                print("Select at least one signal.")
                return

            base = viz_df[
                (viz_df[event_type_col] == w_event.value) &
                (viz_df["session_id"].isin(sel_sessions)) &
                (viz_df[signal_col].isin(sel_signals))
            ]

            # Decide series breakdown
            compare_sessions = bool(w_sess_mode.value)
            compare_signals = bool(w_sig_mode.value)

            series = []

            if compare_sessions and compare_signals:
                # One series per (session, signal)
                for sid in sel_sessions:
                    for sig in sel_signals:
                        sub = base[(base["session_id"] == sid) & (base[signal_col] == sig)]
                        vals = pd.to_numeric(sub[w_metric.value], errors="coerce").dropna().to_numpy()
                        series.append((f"{sid} | {sig}", vals))

            elif compare_sessions and (not compare_signals):
                # One series per session (signals aggregated within session)
                for sid in sel_sessions:
                    sub = base[base["session_id"] == sid]
                    vals = pd.to_numeric(sub[w_metric.value], errors="coerce").dropna().to_numpy()
                    series.append((str(sid), vals))

            elif (not compare_sessions) and compare_signals:
                # One series per signal (sessions aggregated within signal)
                for sig in sel_signals:
                    sub = base[base[signal_col] == sig]
                    vals = pd.to_numeric(sub[w_metric.value], errors="coerce").dropna().to_numpy()
                    series.append((str(sig), vals))

            else:
                # Fully aggregated: sessions + signals all pooled
                vals = pd.to_numeric(base[w_metric.value], errors="coerce").dropna().to_numpy()
                series.append(("aggregate", vals))

            fig, ax = plt.subplots(figsize=(8.3, 4.2))

            for name, vals in series:
                _plot_series(ax, vals, int(w_bins.value), label=(name if (compare_sessions or compare_signals) else None))

            mode_bits = []
            mode_bits.append("sessions=compare" if compare_sessions else "sessions=aggregate")
            mode_bits.append("signals=compare" if compare_signals else "signals=aggregate")

            ax.set_title(
                f"{w_metric.value} distribution\n"
                f"{event_type_col}={w_event.value} | {', '.join(mode_bits)}"
            )
            ax.set_xlabel(w_metric.value)
            if w_cdf.value:
                ax.set_ylabel("Cumulative proportion" if w_norm.value else "Cumulative count")
            else:
                ax.set_ylabel("Proportion" if w_norm.value else "Count")

            ax.grid(True, which="major", axis="both", alpha=0.3)

            if (compare_sessions or compare_signals):
                ax.legend(title="series", fontsize=9)

            if all(len(vals) == 0 for _, vals in series):
                ax.text(0.5, 0.5, "No numeric values after filtering",
                        ha="center", va="center", transform=ax.transAxes)
                ax.set_axis_off()

            plt.show()

            print("Summary stats:")
            for name, vals in series:
                print(_series_stats(name, vals))

    for w in (w_sess_mode, w_sessions, w_sig_mode, w_signals, w_event, w_metric, w_bins, w_cdf, w_norm):
        w.observe(_render, names="value")

    controls = W.VBox([
        W.HBox([w_event, w_metric, w_bins, w_cdf, w_norm]),
        W.HBox([
            W.VBox([w_sess_mode, w_sessions]),
            W.VBox([w_sig_mode, w_signals]),
        ])
    ])

    display(W.VBox([controls, out]))
    _render()

    #return {"viz_df": viz_df, "out": out}
    return {}

# Call explicitly
metric_histogram_widget_v5(
    events_df,
    metrics_df,
    event_type_col="event_name",
    signal_col="signal_col",
    default_bins=10,
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

# Adjust import path as needed
from bodaqs_analysis.segment import extract_segments, SegmentRequest, WindowSpec, RoleSpec

# ---------- Helpers ----------
def _coerce_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple)):
        return list(x)
    if isinstance(x, str):
        return [p.strip() for p in x.split(",") if p.strip()]
    return [x]

def _secondary_time_cols(row):
    """Return list of (name, t_rel) for any *_time_s columns other than canonical ones."""
    out = []
    if row is None:
        return out
    for c in row.index:
        if not isinstance(c, str):
            continue
        if not c.endswith("_time_s"):
            continue
        if c in ("start_time_s", "end_time_s", "trigger_time_s"):
            continue
        v = row.get(c, np.nan)
        if pd.notna(v) and np.isfinite(float(v)):
            out.append((c, float(v)))
    return out

def _series_stats(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return {"n": 0, "min": np.nan, "max": np.nan, "mean": np.nan, "median": np.nan}
    return {
        "n": int(len(x)),
        "min": float(np.min(x)),
        "max": float(np.max(x)),
        "mean": float(np.mean(x)),
        "median": float(np.median(x)),
    }

# ---------- Main widget ----------
def event_browser_widget(
    session: dict,
    schema: dict,
    events_df: pd.DataFrame,
    *,
    default_roles=("disp", "vel"),
    default_pre_s=0.8,
    default_post_s=0.8,
):
    if events_df is None or len(events_df) == 0:
        raise ValueError("events_df is empty")

    # Ensure session meta signals exist (required by segment extractor)
    meta = session.get("meta") or {}
    if "signals" not in meta or not isinstance(meta["signals"], dict) or len(meta["signals"]) == 0:
        raise ValueError("session['meta']['signals'] missing/empty (required for segment extraction)")

    for c in ("session_id", "schema_id", "event_id", "signal_col", "trigger_time_s"):
        if c not in events_df.columns:
            raise ValueError(f"events_df missing required column for browser: {c}")

    # Controls
    sessions = sorted(events_df["session_id"].dropna().astype(str).unique().tolist())
    w_session = W.Dropdown(options=sessions, value=sessions[0], description="Session:", layout=W.Layout(width="380px"))

    w_schema = W.Dropdown(options=[], description="Schema:", layout=W.Layout(width="380px"))
    w_event  = W.Dropdown(options=[], description="Event:", layout=W.Layout(width="520px"))

    w_pre  = W.FloatText(value=float(default_pre_s), description="Pre (s):", layout=W.Layout(width="160px"))
    w_post = W.FloatText(value=float(default_post_s), description="Post (s):", layout=W.Layout(width="160px"))

    # roles you *might* want; you can add more later
    role_options = ["disp", "vel", "acc", "disp_norm", "primary"]
    w_roles = W.SelectMultiple(options=role_options, value=tuple(default_roles), description="Roles:", layout=W.Layout(width="260px", height="120px"))

    w_show_secondary = W.Checkbox(value=True, description="Show secondary triggers")
    w_show_grid = W.Checkbox(value=True, description="Grid")
    w_show_stats = W.Checkbox(value=True, description="Stats")

    out = W.Output()

    def _rebuild_schema_and_events(*_):
        # Filter to selected session
        sub = events_df[events_df["session_id"].astype(str) == str(w_session.value)].copy()
        schema_ids = sorted(sub["schema_id"].dropna().astype(str).unique().tolist())
        if not schema_ids:
            w_schema.options = []
            w_event.options = []
            return

        w_schema.options = schema_ids
        if w_schema.value not in schema_ids:
            w_schema.value = schema_ids[0]

        _rebuild_events()

    def _rebuild_events(*_):
        sub = events_df[
            (events_df["session_id"].astype(str) == str(w_session.value)) &
            (events_df["schema_id"].astype(str) == str(w_schema.value))
        ].copy()

        # Nice labels in dropdown
        labels = []
        for _, r in sub.sort_values("trigger_time_s").iterrows():
            labels.append(f"{r['event_id']}  |  t={float(r['trigger_time_s']):.3f}s  |  {r.get('signal_col','')}")
        w_event.options = labels

        if labels:
            w_event.value = labels[0]

    def _render(*_):
        with out:
            clear_output(wait=True)

            if not w_event.value:
                print("No event selected.")
                return

            # Parse event_id from the label
            event_id = str(w_event.value).split("  |  ", 1)[0].strip()

            # Select exactly one event row
            ev_row_df = events_df[
                (events_df["session_id"].astype(str) == str(w_session.value)) &
                (events_df["schema_id"].astype(str) == str(w_schema.value)) &
                (events_df["event_id"].astype(str) == event_id)
            ].copy()

            if len(ev_row_df) != 1:
                print(f"Expected 1 matching event row, got {len(ev_row_df)}")
                return

            ev_row = ev_row_df.iloc[0]

            # Build a one-row events table and extract a single segment
            events_one = ev_row_df.reset_index(drop=True)

            # Segment request (window override)
            req = SegmentRequest(
                schema_id=str(w_schema.value),
                window=WindowSpec(mode="time", pre_s=float(w_pre.value), post_s=float(w_post.value)),
                roles=[RoleSpec(role=r) for r in w_roles.value if r != "primary"],  # roles resolved via meta['signals']
            )

            bundle = extract_segments(
                session["df"],
                events_one,
                meta=session["meta"],
                schema=schema,
                request=req,
            )

            data = bundle["data"]
            spec = bundle["spec"]
            segs = bundle["segments"]
            evs  = bundle["events"]

            if len(segs) == 0 or not bool(segs.iloc[0]["valid"]):
                print("Segment invalid:", (segs.iloc[0]["reason"] if len(segs) else "no segments"))
                return

            # Time grid
            t_rel = data.get("t_rel_s", None)
            if t_rel is None:
                print("Bundle missing t_rel_s")
                return
            t_rel = np.asarray(t_rel)[0]

            # Build list of series to plot
            series = []

            # Always allow primary, if segment extractor included it
            if "primary" in data and ("primary" in w_roles.value or "primary" not in data.keys()):
                y = np.asarray(data["primary"])[0]
                series.append(("primary", y))

            # Add selected resolved roles
            for r in w_roles.value:
                if r == "primary":
                    continue
                if r in data:
                    y = np.asarray(data[r])[0]
                    series.append((r, y))

            if not series:
                print("No series available to plot (check roles + registry).")
                return

            # Plot
            fig, ax = plt.subplots(figsize=(9.5, 4.2))

            for name, y in series:
                ax.plot(t_rel, y, label=name)

            # Trigger at t_rel=0
            ax.axvline(0.0, linestyle="--", linewidth=1.0)
            ax.text(0.0, 0.98, "trigger", transform=ax.get_xaxis_transform(), ha="left", va="top")

            # Secondary triggers, converted to t_rel by subtracting trigger_time_s
            if w_show_secondary.value:
                t0 = float(ev_row["trigger_time_s"])
                for col, t_abs in _secondary_time_cols(ev_row):
                    tsec = float(t_abs) - t0
                    ax.axvline(tsec, linestyle=":", linewidth=1.0)
                    ax.text(tsec, 0.98, col.replace("_time_s",""), transform=ax.get_xaxis_transform(), rotation=90, ha="right", va="top", fontsize=8)

            ax.set_title(f"Event browser — {ev_row['session_id']} | {ev_row['schema_id']} | {ev_row['event_id']}")
            ax.set_xlabel("t_rel_s (s)")
            ax.set_ylabel("value")

            if w_show_grid.value:
                ax.grid(True, which="both", axis="both", alpha=0.3)

            ax.legend(loc="best")
            plt.show()

            if w_show_stats.value:
                print("Series stats (finite only):")
                for name, y in series:
                    st = _series_stats(y)
                    print(f"  {name:10s}  n={st['n']:4d}  min={st['min']:+.4g}  max={st['max']:+.4g}  mean={st['mean']:+.4g}  median={st['median']:+.4g}")

            # quick spec echo (useful for debugging)
            print("\nResolved spec:")
            print("  anchor:", spec.get("anchor"))
            print("  window:", spec.get("window"))
            print("  role_to_col:", spec.get("role_to_col"))
            print("  qc:", bundle.get("qc"))

    # Wire up
    w_session.observe(_rebuild_schema_and_events, names="value")
    w_schema.observe(_rebuild_events, names="value")

    for w in (w_event, w_pre, w_post, w_roles, w_show_secondary, w_show_grid, w_show_stats):
        w.observe(_render, names="value")

    # initial population + draw
    _rebuild_schema_and_events()
    display(W.VBox([
        W.HBox([w_session, w_schema]),
        W.HBox([w_event]),
        W.HBox([w_pre, w_post, w_show_secondary, w_show_grid, w_show_stats]),
        W.HBox([w_roles]),
        out
    ]))
    _render()

    return {"out": out}

# Usage:
# event_browser_widget(session, schema, events_df)
